# เบื้องหลังหน้าเว็บ: เรียกตลาดด้วย Python

ทุกปุ่มในหน้าเว็บคือ **การเรียก API** หนึ่งครั้ง แบบฝึกนี้ให้ทีม **เขียนการเรียก API เอง** แล้วเล่นตลาดฝึกหนึ่งเกมโดยไม่กดปุ่มในหน้าเว็บ

**วิธีทำ:** แทน `___` ด้วยโค้ดที่ถูก แล้วรันช่องนั้น (Shift+Enter) ทำจากบนลงล่าง · ส่วนที่ไม่มี `___` เขียนไว้ให้แล้ว อ่านแล้วรันได้เลย

| ทำช่วงไหน | ข้อ |
|---|---|
| เช้า 11:30 ทดสอบว่าเครื่องพร้อม | 1–2 |
| บ่าย 14:45–17:00 หลังรอบเก็บคะแนน: เล่นตลาดฝึกด้วย Python (ไม่นับคะแนน) | 3–11 |

**รอบเก็บคะแนนช่วง 1 (13:15–14:45) ขายผ่านหน้าเว็บ `/trade` ด้วยมือเท่านั้น** ห้ามใช้ notebook นี้หรือสคริปต์ใดๆ ส่งคำขอ — ถ้าใช้ คะแนนช่วง 1 เป็น 0 (เซิร์ฟเวอร์บันทึกว่าคำขอมาจากโปรแกรมอะไร) · notebook นี้ใช้กับ **ตลาดฝึก** เท่านั้น

**คำใบ้: ปุ่มไหน เรียก API อะไร**

| ในหน้าเว็บ | API |
|---|---|
| เชื่อมต่อได้ไหม | `GET /v1/version` |
| สถานะ ล็อต คำขอซื้อ | `GET /v1/state` |
| เริ่มใหม่ (ตลาดฝึก) | `POST /v1/sandbox/reset` |
| กล่องสูตรมูลค่า | `GET /v1/calendar` |
| ส่งผลประเมิน | `POST /v1/appraisals` |
| ยื่นซอง | `POST /v1/offers` |
| ข้อเสนอที่เปิดอยู่ | `GET /v1/offers` |
| ผลรอบ | `GET /v1/results` |
| ดูมูลค่าจริง (ตลาดฝึก) | `GET /v1/sandbox/reveal` |

**เขียนอย่างไร:** `GET` = `api.get("/path")` · `POST` = `api.post("/path", json=ข้อมูล)` · ต่อท้าย `.json()` เพื่อได้ dict ของ Python

## 1. ตั้งค่า แล้วเรียก API ครั้งแรก

ใส่ที่อยู่ตลาดฝึกและ key ของทีม (key เดียวกับที่ใส่ในหน้าเว็บ) **อย่าแชร์ key ให้ทีมอื่น**

แล้วเขียนคำขอ **เชื่อมต่อได้ไหม** ต้องได้ `200` และรุ่นของเซิร์ฟเวอร์ · ถ้าขึ้น `No module named 'httpx'` ให้รัน `pip install httpx` ใน Terminal ก่อน

In [ ]:
import httpx

MARKET = "http://<ที่อยู่ตลาด>:8000"   # ทีมงานแจกในห้อง
KEY = "sk-T00-ใส่key-ของทีม"

api = httpx.Client(base_url=MARKET, headers={"Authorization": f"Bearer {KEY}"}, timeout=10)

r = api.get(___)
print(r.status_code, r.json())

## 2. ตลาดตอนนี้เป็นอย่างไร

เขียนคำขอ **สถานะ** ได้ข้อมูลทุกอย่างที่หน้าเว็บแสดง: ช่วงของตลาด (`phase`), เวลาที่เหลือ, ล็อต, คำขอซื้อรอบนี้

| phase | ความหมาย |
|---|---|
| `SETUP` | เตรียมตัว |
| `APPRAISAL` | ส่งผลประเมินได้ |
| `ROUND_OPEN` | รอบเปิดรับข้อเสนอ |
| `ROUND_CLEARING` | กำลังจับคู่ |
| `CLOSED` | ปิดตลาด |

ถ้าเห็นล็อตของทีมครบ 4 ล็อต แปลว่าเครื่องพร้อมสำหรับช่วงบ่ายแล้ว

In [ ]:
state = api.get(___).json()
print("ช่วง:", state["phase"], "· เหลือ", round(state["seconds_left"]), "วินาที")
for lot in state["lots"]:
    print(lot["lot_id"], lot["sample"], "พร้อมขาย", lot["kg_available"] // 60, "กระสอบ")

## 3. เริ่มเกมใหม่ (บ่าย · รอสัญญาณจากผู้สอน)

เหมือนกดปุ่ม **"เริ่มใหม่"** ในหน้าเว็บ แต่เขียนเป็นคำขอเอง ได้ล็อตชุดใหม่ และเวลาประเมิน 10 นาทีเริ่มนับทันที

คำขอนี้ **ส่ง** ข้อมูล ไม่ใช่ขอดู เลือก method ให้ถูก (ไม่ต้องมี body)

In [ ]:
r = api.___(___)
state = r.json()
print(r.status_code, "ช่วง:", state["phase"])
for lot in state["lots"]:
    print(lot["lot_id"], lot["sample"])

## 4. คำนวณมูลค่าด้วย Python (สูตรเดียวกับใน Excel)

ตัวเลขของสูตรอยู่ใน **กล่องสูตรมูลค่า** เขียนคำขอดึงมา แล้วรัน ฟังก์ชันคิด TV เขียนไว้ให้แล้ว

**TV = P0 × G × M × S** ลองเทียบกับ Excel ของทีม ต้องได้เท่ากัน

In [ ]:
rules = api.get(___).json()["rules"]

def grade(d):
    for step in rules["defect_steps"]:          # เรียงจากดีไปแย่ เจอขั้นแรกที่ผ่านก็หยุด
        if d <= step["max_defect_pct"]:
            return step["grade"], step["step"]

def moisture_factor(m):
    mo = rules["moisture"]
    if m <= mo["ref"]:
        return 1.0
    if m <= mo["hard_limit"]:
        return (1 - m / 100) / (1 - mo["ref"] / 100) - mo["penalty_per_pp"] * (m - mo["ref"])
    return mo["hard_factor"]

def screen_factor(s):
    table = {int(k): v for k, v in rules["screen_factors"].items()}
    return table[min(max(s, min(table)), max(table))]

def true_value(sample):
    d, m, s = sample["defect_pct"], sample["moisture_pct"], sample["screen"]
    g = (1 - d / 100) * grade(d)[1]
    return rules["p0"] * g * moisture_factor(m) * screen_factor(s)

tv = {}
for lot in state["lots"]:
    tv[lot["lot_id"]] = round(true_value(lot["sample"]), 2)
    print(lot["lot_id"], grade(lot["sample"]["defect_pct"])[0], tv[lot["lot_id"]], "บาท/กก.")

## 5. ส่งผลประเมินมูลค่า

ส่งได้ **ครั้งเดียว** และเฉพาะตอน `phase` เป็น `APPRAISAL` · `items` เตรียมไว้ให้แล้ว เขียนคำขอส่ง โดย body มีรูปแบบ `{"items": [...]}`

สำเร็จได้ `201` · ถ้าส่งซ้ำได้ error `APPRAISAL_ALREADY_SUBMITTED`

In [ ]:
items = [{"lot_id": lot_id, "est_value_per_kg": value} for lot_id, value in tv.items()]
print(items)

r = api.post(___, json=___)
print(r.status_code, r.json())

## 6. เมื่อมีอะไรผิด ตลาดบอกอะไรกลับมา

ทุก error มีรูปแบบเดียวกัน: `code` (ให้โปรแกรมอ่าน), `message` (ผิดอะไร), `hint` (ควรทำอย่างไร)
ช่องนี้ตั้งใจส่งข้อเสนอผิด: 100 กก. ไม่ใช่ทวีคูณของกระสอบ (60 กก.) อ่านอย่างเดียว รันได้เลย

- ตอนนี้รอบยังไม่เปิด จะได้ `WRONG_PHASE` (ตลาดตรวจช่วงเวลาก่อน)
- ถ้ารันตอนรอบเปิด จะได้ `NOT_MULTIPLE_OF_BAG`

In [ ]:
lot_id = state["lots"][0]["lot_id"]
r = api.post("/v1/offers", json={"rfq_id": "R1-01", "lot_id": lot_id, "kg": 100, "price_per_kg": 120})
print(r.status_code)
print(r.json())

## 7. ดูคำขอซื้อรอบนี้ แล้วหาว่าล็อตไหนผ่านสเปก

รันตอน `phase` เป็น `ROUND_OPEN` (ถ้ายังไม่ถึง รอแล้วรันใหม่) · เขียนคำขอสถานะใหม่ เพราะ RFQ เปลี่ยนทุกรอบ

In [ ]:
state = api.get(___).json()
print("ช่วง:", state["phase"], "· รอบ", (state["round"] or {}).get("number"), "· เหลือ", round(state["seconds_left"]), "วินาที")

def passes(sample, spec):
    return (sample["defect_pct"] <= spec["max_defect_pct"]
            and sample["moisture_pct"] <= spec["max_moisture_pct"]
            and (spec["min_screen"] is None or sample["screen"] >= spec["min_screen"]))

for q in state["rfqs"]:
    ok = [lot["lot_id"] for lot in state["lots"] if lot["kg_available"] >= 60 and passes(lot["sample"], q["spec"])]
    print(q["rfq_id"], q["buyer"], q["buyer_type"], q["bags"], "กระสอบ", "· ล็อตที่ผ่าน:", ok)

## 8. ยื่นซองเสนอขาย 1 รายการ

ตั้ง `RFQ_ID`, `LOT_ID`, `BAGS`, `MARKUP` จากผลข้อ 7 · ราคา = MARKUP × TV · **อย่าต่ำกว่า 0.80** เพราะขายไม่หมดระบบยังให้ 0.80 × มูลค่าจริง

เติม body ให้ครบ 4 ช่อง (`kg` ต้องเป็น กก. ไม่ใช่กระสอบ) แล้วเขียนคำขอยื่นซอง · สำเร็จได้ `201` และ `status` เป็น `OPEN`

In [ ]:
RFQ_ID = state["rfqs"][0]["rfq_id"]     # หรือพิมพ์เอง เช่น "R1-03"
LOT_ID = state["lots"][0]["lot_id"]
BAGS = 2
MARKUP = 1.00

offer = {
    "rfq_id": RFQ_ID,
    "lot_id": LOT_ID,
    "kg": ___,
    "price_per_kg": round(MARKUP * tv[LOT_ID], 1),
}
r = api.post(___, json=___)
print(r.status_code, r.json())

ข้อเสนอที่ส่งด้วย Python ขึ้นในหน้าเว็บด้วย เพราะเป็นตลาดเดียวกัน · เขียนคำขอดู **ข้อเสนอที่เปิดอยู่**

In [ ]:
for o in api.get(___).json()["offers"]:
    print(o["offer_id"], o["rfq_id"], o["lot_id"], o["kg"] // 60, "กระสอบ", o["price_per_kg"], o["status"])

## 9. ดูผลรอบล่าสุด

รันหลังรอบปิด · `ACCEPTED` ขายได้ทั้งหมด · `PARTIAL` บางส่วน · `ABOVE_LIMIT` แพงเกินที่ผู้ซื้อรับได้ · `OUTBID` แพ้ข้อเสนอที่คุ้มกว่า · `SPEC_MISMATCH` คุณภาพจริงไม่ผ่านสเปก

In [ ]:
res = api.get(___).json()
print("รอบ", res["round_id"])
for o in res["my_offers"]:
    print(o["rfq_id"], o["lot_id"], o["kg"] // 60, "กระสอบ", o["price_per_kg"], "→", o["status"], "ขายได้", o["filled_kg"] // 60)

## 10. รอบถัดไป: ยื่นทุก RFQ ในครั้งเดียว

รอบต่อๆ ไปทำซ้ำข้อ 7 → 8 → 9 ได้ แต่จะช้า ช่องนี้วนยื่นให้ทุก RFQ ที่มีล็อตผ่านสเปก
ตั้ง MARKUP แยกตามประเภทผู้ซื้อเอง (ดูผลข้อ 9 ประกอบ) แล้วเติม body ของซอง · รันใหม่ทุกครั้งที่รอบเปิด

In [ ]:
MARKUP_BY_TYPE = {"COMMERCIAL": ___, "INSTANT": ___}

state = api.get("/v1/state").json()
print("รอบ", (state["round"] or {}).get("number"), "· เหลือ", round(state["seconds_left"]), "วินาที")
for q in state["rfqs"]:
    ok = [lot for lot in state["lots"] if lot["kg_available"] >= 60 and passes(lot["sample"], q["spec"])]
    if not ok:
        continue
    lot = ok[0]
    bags = min(q["bags"], lot["kg_available"] // 60)
    price = round(MARKUP_BY_TYPE[q["buyer_type"]] * tv[lot["lot_id"]], 1)
    offer = ___
    r = api.post("/v1/offers", json=offer)
    print(q["rfq_id"], lot["lot_id"], bags, "กระสอบ", price, "→", r.status_code, r.json().get("status") or r.json()["error"]["code"])
    lot["kg_available"] -= bags * 60      # กันยื่นกระสอบเดียวกันซ้ำในรอบเดียว

## 11. จบ 6 รอบ: ดูมูลค่าจริง

รันเมื่อ `phase` เป็น `CLOSED` · เหมือนปุ่ม **"ดูมูลค่าจริง"** ในหน้าเว็บ

In [ ]:
rv = api.get(___).json()
for lot in rv["lots"]:
    est = (lot["your_estimate"] or {}).get("est_value_per_kg")
    print(lot["lot_id"], "TV จริง", lot["true_value_per_kg"], "· เราประเมิน", est)
sc = rv["score"]
print("PR", sc["pr"], "· คะแนนประเมิน", round(sc["appraisal_score"], 1), "· คะแนนรวม", round(sc["total"], 1))

## ก้าวต่อไป (พรุ่งนี้): ให้โปรแกรมทำเองทุกรอบ

ทุกอย่างข้างบนคือสิ่งที่ agent ทำ ต่างกันแค่ agent **วนซ้ำเอง** ทุกรอบ:

```
ทุก 1–2 วินาที:
    ดูสถานะ (GET /v1/state)
    ถ้าถึงช่วงประเมินและยังไม่ส่ง → คำนวณ TV แล้วส่ง
    ถ้ารอบใหม่เปิด → ตัดสินใจว่าจะเสนอใคร เท่าไร ราคาเท่าไร → ส่ง
    ถ้ารอบปิด → อ่านผล จำไว้ใช้รอบหน้า
```

ส่วน "ตัดสินใจ" คือที่ใช้ทั้งกฎ (Python) และ LLM เรียนกันวันที่ 18
**ห้ามเรียก API ถี่เกิน 5 ครั้งต่อวินาที** (จะได้ error `RATE_LIMITED`)